In [3]:
#cell 1 导入库和配置参数
import pandas as pd
import json
import os
import time
from openai import OpenAI
from tqdm.notebook import tqdm # 使用 notebook 专用的进度条，显示更漂亮

# ================= 配置区 =================

# 1. 填入你的 DeepSeek API Key
API_KEY = "" 

# 2. DeepSeek 官方配置
BASE_URL = "https://api.deepseek.com"
MODEL_NAME = "deepseek-chat"

# 3. 文件路径
INPUT_FILE = '../data/processed/to_be_distilled.csv'
OUTPUT_FILE = '../data/processed/train_dataset_raw.jsonl'

# 初始化 OpenAI 客户端 (DeepSeek 兼容 OpenAI 格式)
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

print("环境配置完成！")

环境配置完成！


In [4]:
#Cell 2: 定义 Prompt (核心指令)
# === 核心 System Prompt ===
SYSTEM_PROMPT = """
You are an expert Cardiologist and Clinical Data Specialist.
Your task is to extract structured findings from the ECG report text provided by the user.

Output strictly in JSON format. Do not output markdown code blocks (```json), just the raw JSON string.

### Schema Definitions
1. **type**:
   - `RHYTHM`: Timing/Conduction issues (e.g., Sinus rhythm, AFib, Blocks).
   - `MORPHOLOGY`: Wave shape/appearance (e.g., ST elevation, T wave inversion, High voltage, Q waves).
   - `DIAGNOSIS`: Clinical conclusion/Disease (e.g., Myocardial Infarction, Hypertrophy, Ischemia).

2. **standard_name**: The normalized clinical term.
   - **CRITICAL RULE**: Remove anatomical locations (e.g., "Anterior MI" -> "Myocardial Infarction").
   - **CRITICAL RULE**: KEEP severity or degree in the name (e.g., KEEP "First Degree AV Block", "Type II Block").

3. **meta**:
   - `location`: List of anatomical locations (e.g., ["Inferior", "V1-V3"]).
   - `acuity`: "ACUTE", "CHRONIC" (if text says "old"), or "UNKNOWN".
   - `confidence`: "HIGH" (default), "LOW" (if "possible", "suggests"), "NEGATIVE" (if "no signs of").

### Example Input
"Sinus tachycardia. Possible old inferior myocardial infarction. First degree AV block."

### Example Output
{
  "findings": [
    {
      "original_text": "Sinus tachycardia",
      "type": "RHYTHM",
      "standard_name": "Sinus Tachycardia",
      "meta": {"location": [], "acuity": "UNKNOWN", "confidence": "HIGH"}
    },
    {
      "original_text": "old inferior myocardial infarction",
      "type": "DIAGNOSIS",
      "standard_name": "Myocardial Infarction",
      "meta": {"location": ["Inferior"], "acuity": "CHRONIC", "confidence": "LOW"}
    },
    {
      "original_text": "First degree AV block",
      "type": "RHYTHM",
      "standard_name": "First Degree AV Block",
      "meta": {"location": [], "acuity": "UNKNOWN", "confidence": "HIGH"}
    }
  ]
}
"""
print("Prompt 定义完成！")

Prompt 定义完成！


In [5]:
#Cell 3: 定义 API 调用函数
def get_deepseek_response(text):
    """调用 DeepSeek API 进行抽取"""
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Input Report: \"{text}\"\nJSON Output:"}
            ],
            temperature=0.1, # 低温以保证格式稳定
            max_tokens=1024,
            response_format={ "type": "json_object" } # 强制返回 JSON 对象
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"\n[API Error]: {e}")
        time.sleep(2) # 出错歇一会再重试
        return None

print("API 函数定义完成！")

API 函数定义完成！


In [7]:
#Cell 4: 主逻辑 - 开始批量蒸馏 (Batch Process)
# 1. 读取数据
if not os.path.exists(INPUT_FILE):
    print(f"❌ 错误：找不到输入文件 {INPUT_FILE}")
else:
    df = pd.read_csv(INPUT_FILE)
    total_count = len(df)
    
    # 2. 断点续传检查
    processed_count = 0
    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
            processed_count = sum(1 for _ in f)
    
    print(f"📊 总数据量: {total_count}")
    print(f"✅ 已处理: {processed_count}")
    
    if processed_count >= total_count:
        print("🎉 所有数据已处理完毕！无需运行。")
    else:
        print(f"🚀 即将开始处理剩余的 {total_count - processed_count} 条数据...")
        
        # 切片，只跑剩下的
        df_to_process = df.iloc[processed_count:]

        # 打开文件准备追加写入 ('a' 模式)
        with open(OUTPUT_FILE, 'a', encoding='utf-8') as f_out:
            
            # 启动进度条
            for idx, row in tqdm(df_to_process.iterrows(), total=len(df_to_process), desc="DeepSeek Distilling"):
                report_text = row['report_text']
                
                # API 调用
                json_result = get_deepseek_response(report_text)
                
                if json_result:
                    try:
                        # 校验 JSON 是否合法
                        parsed = json.loads(json_result)
                        
                        # 构造训练数据格式
                        train_entry = {
                            "instruction": "Extract structured findings from the ECG report.",
                            "input": report_text,
                            "output": json_result
                        }
                        
                        # 写入文件
                        f_out.write(json.dumps(train_entry, ensure_ascii=False) + "\n")
                        f_out.flush() # 实时保存
                        
                    except json.JSONDecodeError:
                        print(f"\n[Json Error] 返回了无效 JSON，跳过...")
                else:
                    print(f"\n[Skip] API 无响应，跳过...")

        print(f"\n✨ 任务完成！结果已保存至: {OUTPUT_FILE}")

📊 总数据量: 2000
✅ 已处理: 6
🚀 即将开始处理剩余的 1994 条数据...


DeepSeek Distilling:   0%|          | 0/1994 [00:00<?, ?it/s]


✨ 任务完成！结果已保存至: ../data/processed/train_dataset_raw.jsonl


In [12]:
#cell 5 验证结果
# 读取生成文件的前 3 行看看效果
print("=== 预览生成的数据集 (前3条) ===")
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
        for i in range(3):
            line = f.readline()
            if not line: break
            data = json.loads(line)
            print(f"\n[样本 {i+1}]")
            print(f"Input: {data['input']}")
            print(f"Output: {data['output'][:150]} ...") # 只打印前150个字符避免刷屏
else:
    print("文件还没生成呢，先跑上面的 Cell 吧！")

=== 预览生成的数据集 (前3条) ===

[样本 1]
Input: anteroseptal myocardial infarction, inferior myocardial infarction, abnormal QRS, sinus bradycardia
Output: {
  "findings": [
    {
      "original_text": "anteroseptal myocardial infarction",
      "type": "DIAGNOSIS",
      "standard_name": "Myocardial Inf ...

[样本 2]
Input: first degree AV block, sinus rhythm
Output: {
  "findings": [
    {
      "original_text": "first degree AV block",
      "type": "RHYTHM",
      "standard_name": "First Degree AV Block",
       ...

[样本 3]
Input: complete left bundle branch block, first degree AV block, sinus rhythm
Output: {
  "findings": [
    {
      "original_text": "complete left bundle branch block",
      "type": "RHYTHM",
      "standard_name": "Left Bundle Branch ...
